**New benchmark analysis (2026-10-05).** Uses the available 4L/2H/K16 checkpoints with weight decay **0.1**, seeds 100–104. This is a new exploratory analysis, not reproduction of SEL-4L wd0.01. References/conflict definitions are frozen historical inputs, verified by hash. All model-dependent outputs are recomputed. Validation match splits support the stated comparisons; prior test access prevents a fresh confirmatory claim. Population and intervention scope apply to each reported result. See `config/analysis_plan.json` and `SURGICAL_FIXES.md`.

# MB11b — Causal Scrubbing of the Proposed Mechanism

**Track B · Mechanistic interpretability · Identity × Context project**

| Item | Content |
|---|---|
| **Question** | Does the *proposed mechanism* — not merely the set of components, but the claim about what each component computes — account for the behaviour? |
| **Purpose** | MB11 showed whether the circuit is sufficient when everything else is mean-ablated. That test is blind to what the circuit's parts are supposed to be doing: it would pass equally for a circuit whose components were doing something entirely different. Causal scrubbing closes that gap. Each component is resampled from *within the equivalence class its claimed role allows*; if the claim is right, the behaviour survives, and if the claim is wrong, it does not. |
| **Inputs** | `config/circuit.json`, `config/mechanism_hypothesis.json` (the researcher's abstraction), `artifacts/populations/*`, SEL-4L primary seed |
| **Outputs** | `results/MB11b_*`, `config/mechanism_hypothesis.json` |
| **Model under study** | SEL-4L, primary seed (replicates are covered by MB11) |
| **Test data** | none — the test split is opened only in MB12 |
| **Modules used** | `circuits.scrub_edits`, `circuits.class_donors` (written in MB07) |

Run after MB11 and before MB12.

> **⚠ REPAIR STATUS (2026-10-04).** Review findings **F02, F03, F04, F07** are confirmed against this notebook's code.
> * **F02** — `scrub_measures` (Section 3) runs the donor forward pass once, on the recorded donor, and writes those outputs
>   into **both** the recorded run and the player-removed run. Every scrubbed component then gives the same output in both
>   runs, so the identity effect is blocked by construction, whatever the hypothesis. The saved identity-effect recoveries
>   (≈ 0.000) are **invalid as a test of the abstraction**. The arbitration rows measure Δ with fixed donor outputs, a
>   narrower question than the paired effect.
> * **F03** — the self-check compares only the recorded run; `corrupt_self` is computed and never checked.
> * **F04** — the wrong-class control permutes `a_ctx`, `team_change` and `player` **independently** in a copy of the table
>   that is then also used for scoring. Δ is therefore read for a different action pair, and joint class sizes are not preserved.
> * **F07** — every `computes` sentence in `config/mechanism_hypothesis.json` is empty. The file is kept as it is (it is
>   fingerprinted and locked by MB12). New hypotheses written now are labelled **post-hoc**.
>
> The defective legacy experiment sections are disabled in this new benchmark. **Section R** re-runs the experiment with paired
> counterfactual donors (`src/scrub.py`), checks all three self-scrub invariants, uses a wrong-class control that permutes
> complete class labels and never touches the scoring columns, and tests post-hoc competing hypotheses side by side.

In [1]:
import os
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = os.environ.get("MI_CPU_THREADS", "4")
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: True | RUN_MODE: full
ROOT: /content/soccer_mi
Track A (read only): /content/soccer_mi/model_training_evaluation | exists: True
Track B: /content/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.patching as patching
import mi_src.circuits as circuits
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, patching, circuits]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(int(os.environ.get("MI_CPU_THREADS", "4")))
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)

Device: cuda | outputs: /content/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
verified = fingerprint.verify_files(MANIFEST)
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])

N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]
DRAWS = SETTINGS["scrub_draws"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads

circuit = mi_setup.load_json(OUT / "config" / "circuit.json")
candidates = mi_setup.load_json(OUT / "config" / "candidates.json")
CIRCUIT = circuit["components"]
ALL = circuits.all_components(N_LAYERS, N_HEADS)
OUTSIDE = [c for c in ALL if c not in CIRCUIT]
p_player = val_conflicts[val_conflicts.population == "P-player"].reset_index(drop=True)
estimation = p_player[p_player.half == "estimation"].reset_index(drop=True)
RUN = len(CIRCUIT) > 0 and len(estimation) > 0
print(f"verified files: {len(verified)} | primary seed {PRIMARY_SEED} | identity variable: {IDENTITY}")
print(f"circuit: {CIRCUIT} | outside the circuit: {OUTSIDE}")
print(f"estimation-half P-player rows: {len(estimation):,} | scrub draws: {DRAWS}")
if not RUN:
    print("BLOCKED-SOFT: the circuit is empty, so there is no mechanism to scrub; every section below is skipped.")

/content/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 65 | primary seed 103 | identity variable: player
circuit: ['L0H0', 'L0H1', 'L0MLP', 'L1H0', 'L1H1', 'L1MLP', 'L2H0', 'L2H1', 'L2MLP', 'L3H1', 'L3MLP'] | outside the circuit: ['L3H0']
estimation-half P-player rows: 529 | scrub draws: 5


## 1. Terms

| Term | Meaning here |
|---|---|
| **Abstraction** | The researcher's claim about what each circuit component computes — in words, before any number is produced. "L1H0 reads the acting player's own earlier events" is an abstraction; "L1H0 matters" is not. |
| **Equivalence class** | The set of events the abstraction says a component *cannot tell apart*. If L1H0 only reads who is acting, then two events with the same acting player are equivalent for it, and swapping its output between them must not change the behaviour. |
| **Scrub** | Replacing a component's output with the output it had on a different event. Scrubbing **within** a class tests the abstraction; scrubbing **across** everything tests nothing but removal. |
| **Hypothesis-consistent scrub** | Every circuit component resampled inside its own class, every non-circuit component resampled from anywhere. This is the run the hypothesis predicts will still behave. |
| **Floor** | Every component resampled from anywhere. Nothing of the mechanism survives; this is what "explains nothing" looks like on the same scale. |
| **Recovered share** | Pooled, sign-aligned, (scrubbed − floor) / (full − floor). 1 = the scrub behaves exactly like the intact model; 0 = like the floor. The blueprint's bar is ≥ 0.80 for the hypothesis-consistent scrub and ≤ 0.20 when the circuit itself is scrubbed away (§23). |

**Why this is stronger than MB11.** MB11's resample ablation draws replacements from anywhere, which
is the *floor* run below. The only thing that distinguishes causal scrubbing from it is the class
structure — so the comparison between the hypothesis-consistent scrub and the wrong-class control in
Section 5 is the whole experiment. If the two agree, the abstraction has no content.

## 2. The abstraction — written by the researcher before any number

`config/mechanism_hypothesis.json` holds one entry per circuit component: what it is claimed to
compute, and the event fields that define its equivalence class. The cell below writes a **default**
derived from which MB06 contrast selected each component — identity-selected components are claimed
to depend on who is acting, context-selected ones on the situation — and then stops overwriting the
file. The default is a starting point, not a hypothesis: the researcher should open the file, write
the `computes` sentences in plain words, and choose `class_keys` that actually express the claim.

Available class keys are the event fields the conflict set carries: `player`, `team`, `role`,
`a_ctx`, `a_id`, `zone`, `minute_bin`, `team_change`, `coupled_with_prev`, `prev_action_id`.

**A narrower class is a stronger claim.** `["player"]` says the component carries nothing but the
acting player's identity. `["player", "zone"]` is weaker — it allows the component to depend on
location too — and will therefore recover more. Report which classes were used alongside the
recovered share, because the number means nothing without them.

> **⚠ REPAIR NOTE (2026-10-04) — HYPOTHESIS GAP (documented, not hidden) — F07.** This cell wrote a DEFAULT abstraction with empty `computes` sentences. That file is kept unchanged; it is fingerprinted, and MB12's lock records its SHA-256. Writing sentences into it now would make a post-hoc statement look pre-registered. Section R5 writes `config/mechanism_hypotheses_posthoc.json`, which is labelled post-hoc.

In [4]:
CLASS_KEYS_AVAILABLE = ["player", "team", "role", "a_ctx", "a_id", "zone", "minute_bin",
                       "team_change", "coupled_with_prev", "prev_action_id"]
CLASS_KEYS_AVAILABLE = [key for key in CLASS_KEYS_AVAILABLE if key in estimation.columns]

selected_by = {}
for item in candidates["candidates"]:
    label = item["site"]
    if "head" in label:
        name = f"L{label[1]}H{label[-1]}"
    elif "mlp_out" in label:
        name = f"L{label[1]}MLP"
    elif "attn_out" in label:
        name = None
    else:
        name = None
    targets = [f"L{label[1]}H{h}" for h in range(N_HEADS)] if "attn_out" in label else ([name] if name else [])
    for target in targets:
        selected_by.setdefault(target, set()).add(item["contrast"])

DEFAULT_KEYS = {"identity removal": ["player"], "context": ["a_ctx", "team_change"]}
hypothesis_path = OUT / "config" / "mechanism_hypothesis.json"
if hypothesis_path.exists():
    hypothesis = mi_setup.load_json(hypothesis_path)
    print("mechanism hypothesis loaded from", hypothesis_path.name, "- not overwritten")
else:
    nodes = {}
    for component in CIRCUIT:
        contrasts = sorted(selected_by.get(component, {"identity removal"}))
        keys = []
        for contrast in contrasts:
            for key in DEFAULT_KEYS.get(contrast, []):
                if key in CLASS_KEYS_AVAILABLE and key not in keys:
                    keys.append(key)
        nodes[component] = {"computes": "",
                            "class_keys": keys or ["player"],
                            "selected_by": contrasts}
    hypothesis = {"nodes": nodes,
                  "outside_circuit": "resampled without restriction: the hypothesis claims these carry "
                                     "nothing about the identity-context arbitration",
                  "scrub_scope": "target",
                  "note": "DEFAULT written by MB11b. Fill in every 'computes' sentence and choose "
                          "'class_keys' that express it before reading Section 5 as a result.",
                  "written_at": fingerprint.now()}
    mi_setup.save_json(hypothesis_path, hypothesis)
    print("default mechanism hypothesis written to", hypothesis_path.name)
fingerprint.record_files(MANIFEST, {"config/mechanism_hypothesis.json": hypothesis_path})

CLASS_KEYS = {component: hypothesis["nodes"][component]["class_keys"] for component in CIRCUIT
              if component in hypothesis["nodes"]}
AT_TARGET = hypothesis.get("scrub_scope", "target") == "target"
unfilled = [c for c, node in hypothesis["nodes"].items() if not node.get("computes")]
print()
print(pd.DataFrame([{"component": c, "class keys": ", ".join(CLASS_KEYS.get(c, [])),
                     "computes": hypothesis["nodes"].get(c, {}).get("computes", "") or "(not written yet)"}
                    for c in CIRCUIT]).to_string(index=False))
if unfilled:
    print(f"\n{len(unfilled)} component(s) have no 'computes' sentence yet: {unfilled}")
    print("The numbers below will still be produced, but they do not test a stated abstraction until it is written.")
for component in CIRCUIT:
    sizes = estimation.groupby(CLASS_KEYS.get(component, ["player"])).size()
    print(f"{component}: {len(sizes)} classes | singleton classes {int((sizes == 1).sum())} "
          f"| median class size {float(sizes.median()):.1f}")

default mechanism hypothesis written to mechanism_hypothesis.json

component                 class keys          computes
     L0H0 a_ctx, team_change, player (not written yet)
     L0H1                     player (not written yet)
    L0MLP a_ctx, team_change, player (not written yet)
     L1H0         a_ctx, team_change (not written yet)
     L1H1         a_ctx, team_change (not written yet)
    L1MLP a_ctx, team_change, player (not written yet)
     L2H0                     player (not written yet)
     L2H1                     player (not written yet)
    L2MLP a_ctx, team_change, player (not written yet)
     L3H1         a_ctx, team_change (not written yet)
    L3MLP a_ctx, team_change, player (not written yet)

11 component(s) have no 'computes' sentence yet: ['L0H0', 'L0H1', 'L0MLP', 'L1H0', 'L1H1', 'L1MLP', 'L2H0', 'L2H1', 'L2MLP', 'L3H1', 'L3MLP']
The numbers below will still be produced, but they do not test a stated abstraction until it is written.
L0H0: 243 classes | singl

## 3. Built-in checks before the scrub

1. **Self-scrub.** If every donor is the event itself, every activation is replaced by its own
   value, so Delta must be unchanged to floating-point precision. This catches a wrong position
   index, which is the easiest way to get a silently meaningless scrub.
2. **Singleton classes.** A class with one member can only donate to itself, so those events are
   not really scrubbed. Their share is printed above and reported with every result; if it is
   large, the class is too narrow for this population and the recovered share is inflated.

> **⚠ REPAIR NOTE (2026-10-04) — INVALIDATED — IMPLEMENTATION BUG / VERIFICATION GAP — F02, F03.** `scrub_measures` writes the recorded donor's activation into both runs (F02). The self-check asserts only `clean_self` (F03); applied to `corrupt_self` it would fail, because the player-removed run receives recorded-run activations. "self-scrub passes" here is not evidence that the paired experiment is implemented correctly. Kept to reproduce the historical numbers. Corrected: Section R1.

In [5]:
@torch.no_grad()
def scrub_measures(assignment, table, at_target=True):
    """Delta on the recorded event and under identity removal, with the given scrub applied to both."""
    clean_out = np.empty(len(table))
    corrupt_out = np.empty(len(table))
    components = list(assignment)
    sites = circuits.component_sites(components) if components else []
    for part, inputs, positions, a_ctx, a_id in mi_setup.event_batches(bundle, val_view, table, BATCH):
        chunk = table.iloc[part]
        replace = {}
        if components:
            donor_index = {c: assignment[c][part] for c in components}
            needed = sorted({i for values in donor_index.values() for i in np.unique(values)})
            lookup = {row: position for position, row in enumerate(needed)}
            donor_table = table.iloc[needed]
            donor_inputs, _ = mi_setup.window_inputs(bundle, val_view, donor_table["window"].to_numpy())
            _, donor_cache = hooks.forward(MODEL, donor_inputs, capture=sites)
            donor_positions_all = torch.as_tensor(donor_table["position"].to_numpy(), device=DEVICE)
            per_site = {}
            for component in components:
                kind, layer, head = circuits.parse(component)
                site = f"head_z_{layer}" if kind == "head" else f"mlp_out_{layer}"
                take = torch.as_tensor([lookup[i] for i in donor_index[component]], device=DEVICE)
                source = donor_cache[site][take]
                source_positions = donor_positions_all[take]
                if at_target:
                    edit = circuits.copy_from(source, source_positions, positions, head=head)
                else:
                    edit = hooks.set_values(source, positions=None, head=head)
                per_site.setdefault(site, []).append(edit)
            replace = {site: hooks.combine(*edits) for site, edits in per_site.items()}
        removed = interventions.remove_feature(inputs, IDENTITY, positions)
        clean_out[part] = circuits.delta_under(MODEL, inputs, positions, a_ctx, a_id, replace)
        corrupt_out[part] = circuits.delta_under(MODEL, removed, positions, a_ctx, a_id, replace)
    return clean_out, corrupt_out


if RUN:
    identity_assignment = {component: np.arange(len(estimation)) for component in CIRCUIT}
    clean_self, corrupt_self = scrub_measures(identity_assignment, estimation, AT_TARGET)
    clean_intact, corrupt_intact = circuits.identity_effects(
        MODEL, bundle, val_view, estimation, BATCH,
        lambda inputs, positions, chunk: interventions.remove_feature(inputs, IDENTITY, positions))
    largest = float(np.abs(clean_self - clean_intact).max())
    print(f"self-scrub: largest change in Delta {largest:.3e} (must be ~0)")
    assert largest < 1e-4, "BLOCKED: scrubbing an event with its own activation changed Delta."
    singleton = {}
    for component in CIRCUIT:
        sizes = estimation.groupby(CLASS_KEYS.get(component, ["player"]))["event_row"].transform("size")
        singleton[component] = float((sizes == 1).mean())
    print("share of events in a singleton class (not really scrubbed):",
          {k: round(v, 3) for k, v in singleton.items()})
else:
    singleton = {}
    print("skipped")

self-scrub: largest change in Delta 0.000e+00 (must be ~0)
share of events in a singleton class (not really scrubbed): {'L0H0': 0.229, 'L0H1': 0.0, 'L0MLP': 0.229, 'L1H0': 0.006, 'L1H1': 0.006, 'L1MLP': 0.229, 'L2H0': 0.0, 'L2H1': 0.0, 'L2MLP': 0.229, 'L3H1': 0.006, 'L3MLP': 0.229}


## 4. The four runs

| Run | Circuit components | Everything else | What it means |
|---|---|---|---|
| **full** | untouched | untouched | the model |
| **hypothesis** | resampled **within** each component's class | resampled freely | the mechanism as stated |
| **circuit scrubbed** | resampled **freely** | untouched | the mechanism removed (completeness) |
| **floor** | resampled freely | resampled freely | nothing survives |

Both behaviours of MB11 are measured: the **identity effect** (Delta on the recorded event minus
Delta with the player removed, with the same scrub applied to both runs) and **arbitration**
(Delta itself). Each scrub is averaged over `scrub_draws` independent donor draws, and every
quantity is a pooled sign-aligned ratio with a match-level bootstrap interval.

> **⚠ REPAIR NOTE (2026-10-04) — INVALIDATED — IMPLEMENTATION BUG — F02.** Identity-effect rows: blocked by construction, so they cannot test the abstraction. Arbitration rows: Δ with donor outputs fixed at the scored position. Valid as that narrower measurement, but they cannot validate preservation of the identity effect. Corrected: Section R2.

In [6]:
print("Obsolete defective historical experiment archived; execute corrected Section R below.")


Obsolete defective historical experiment archived; execute corrected Section R below.


## 5. The control that gives the experiment its meaning — wrong classes

The hypothesis-consistent scrub is only informative next to a scrub with the **same amount of
resampling but the wrong class structure**. Each component's class labels are permuted across
events, ~~which preserves the number of classes and their sizes exactly while destroying their
meaning~~. *[Corrected 2026-10-04 (F04): the code below permutes each key column independently in a table that is also used for scoring. That changes the scored action pair and does not preserve joint class sizes. See Section R3.]*

* hypothesis ≫ wrong-class: the abstraction carries real content — the component really is
  insensitive to what its class says it is insensitive to.
* hypothesis ≈ wrong-class: the recovered share is an artefact of how much resampling happened,
  not of the claim. The abstraction is then unsupported, **whatever** its absolute value.

A second control scrubs each component within a class built from a field the abstraction does *not*
name, which shows how much any class structure of similar granularity would recover.

> **⚠ REPAIR NOTE (2026-10-04) — INVALIDATED — EXPERIMENT DESIGN + IMPLEMENTATION BUG — F04, F02.** The shuffled table is used for scoring, so `a_ctx` is permuted; joint class sizes are not preserved; and the scrub inherits F02. Corrected: Section R3.

In [7]:
print("Obsolete defective historical experiment archived; execute corrected Section R below.")


Obsolete defective historical experiment archived; execute corrected Section R below.


## 6. Which node's abstraction is doing the work

One node at a time is scrubbed **freely** while every other circuit node stays
hypothesis-consistent. A node whose free scrub barely lowers the recovered share has an abstraction
that is not load-bearing here: either it carries nothing, or something else supplies the same
information. The drop is the per-node analogue of MB11's minimality, but about the *claim* rather
than about the component's presence.

> **⚠ REPAIR NOTE (2026-10-04) — INVALIDATED — IMPLEMENTATION BUG — F02.** Per-node drops inherit F02 (the identity effect is blocked in every run). Corrected: Section R4.

In [8]:
print("Obsolete defective historical experiment archived; execute corrected Section R below.")


Obsolete defective historical experiment archived; execute corrected Section R below.


In [9]:
print("Obsolete defective historical experiment archived; execute corrected Section R below.")


Obsolete defective historical experiment archived; execute corrected Section R below.


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- Why it was tested:
- What was observed:
- What the result supports:
- What it contradicts:
- Alternative explanations that remain:
- What it does not establish:
- Evidence label (Observed / Supported / Strong causal evidence / Hypothesized):

In [10]:
print("Obsolete defective historical experiment archived; execute corrected Section R below.")


Obsolete defective historical experiment archived; execute corrected Section R below.


## 7. What this scrub does and does not establish

* It tests the abstraction **as written in `mechanism_hypothesis.json`**. A vague abstraction with
  wide classes is easy to satisfy; the class sizes printed in Section 2 are part of the result.
* The scrub replaces activations at the **scoring position** (`scrub_scope: "target"`). A component
  that does its work at an earlier position is therefore only partly scrubbed. Setting
  `scrub_scope` to `"all"` in the hypothesis file takes the donor's whole sequence instead, which is
  a stronger scrub but puts the donor's entire history into the run.
* Donors come from the same estimation-half population, so a class with few members in that
  population gives a weak scrub; the singleton share quantifies it.
* Nothing here is held out. MB12 repeats the hypothesis-consistent scrub once on the test split,
  against a prediction written before the split is opened.

## R. The corrected causal scrub (REPAIR 2026-10-04)

### What the scrub tests, stated formally

The behaviour is the **paired** identity effect E(x) = Δ(x) − Δ(φ(x)), where φ removes the player at the scored position.
The hypothesis gives each circuit component *c* class keys *k_c* and claims out_c(x) = g_c(k_c(x)). Components outside the
circuit are claimed to carry nothing about the behaviour (free).

### How donors are constructed, and why (the decision)

For each event *x* and component *c*, one donor *d* is drawn from *c*'s class of *x* (another event when the class has one).

* **The player is one of c's keys** → *c* may respond to the player. The player-removed run receives *c*'s output on **φ(d)**:
  the same donor event with the player removed at the donor's own target. Under the hypothesis this is exact, because φ
  changes *d*'s key tuple exactly as it changes *x*'s, so φ(d) is in the class of φ(x).
* **The player is not one of c's keys (or c is free)** → the hypothesis says *c* ignores the player. The player-removed run
  receives the **same** activation, *c*'s output on *d*.

Both are exact samples from the hypothesis's equivalence classes. One donor per event, paired across the two runs, is used
deliberately. The effect is about 0.17 nats, while Δ varies by several nats between donors. Independent donors for the
two runs would make each per-event effect mostly donor noise, and sign-aligned recovery then reads that noise as recovery.
The R2 noise check measures this; MB16's design shows it (finding F25).

MB11b's original design used "same" for every component, including those whose keys contain the player (F02).
It is reproduced below **with the same donors** so the difference is due to the branch rule alone.

### Built-in checks (F03)

With every donor equal to the event itself and every component paired, the scrubbed model must reproduce (1) the recorded
run, (2) the player-removed run and (3) their difference, each to floating-point precision. All three are asserted.

In [11]:
import importlib
try:
    import mi_src.scrub as scrub
    import mi_src.steering as steering
    import mi_src.context_edits as context_edits
    import mi_src.graph as graph
except ModuleNotFoundError as error:
    raise RuntimeError("BLOCKED: the repaired modules (scrub, steering, context_edits, graph) are missing. Run "
                       "MB07R_repaired_sources_and_regression_tests.ipynb, then restart the kernel.") from error
for module in [hooks, mi_metrics, circuits, scrub, steering, context_edits, graph]:
    importlib.reload(module)
for module in [circuits, mi_metrics, scrub, steering, context_edits, graph]:
    if getattr(module, "REPAIR_VERSION", None) != "2026-10-04":
        raise RuntimeError(f"BLOCKED: {module.__name__} on disk is not the repaired 2026-10-04 version. Re-run the repaired "
                           "%%writefile cells (MB02 for mi_metrics, MB07 for circuits, MB07R for the new modules), then restart.")
print("repaired modules loaded:", [m.__name__ for m in [circuits, mi_metrics, scrub, steering, context_edits, graph]])

def remove_identity(inputs, positions):
    return interventions.remove_feature(inputs, IDENTITY, positions)


if RUN:
    check = scrub.self_scrub_check(MODEL, bundle, val_view, estimation, ALL, remove_identity, BATCH, AT_TARGET)
    for name, value in check["errors"].items():
        print(f"self-scrub, {name}: largest |error| {value:.2e}")
    print(f"for comparison, the original same-donor design: player-removed-run error {check['original design (same donor in both runs): counterfactual-run error']:.3f}, "
          f"effect error {check['original design: effect error']:.3f}")
    assert check["passed"], "BLOCKED: the paired scrub does not reproduce the intact model when donors are the events themselves"

repaired modules loaded: ['mi_src.circuits', 'mi_src.mi_metrics', 'mi_src.scrub', 'mi_src.steering', 'mi_src.context_edits', 'mi_src.graph']


self-scrub, clean run vs intact clean: largest |error| 0.00e+00
self-scrub, counterfactual run vs intact counterfactual: largest |error| 0.00e+00
self-scrub, effect vs intact effect: largest |error| 0.00e+00
for comparison, the original same-donor design: player-removed-run error 3.437, effect error 3.437


### R2 — The four runs, corrected

Same runs as Section 4, same class keys (those in `config/mechanism_hypothesis.json`), same `scrub_draws`. Added: the
original same-donor rule on **identical donors** (isolates F02), and a second independent floor draw (the noise check: its
recovered share against the first floor must be near 0).

In [12]:
def run(spec, seed, **options):
    clean, counter, info = scrub.averaged_scrub(MODEL, bundle, val_view, estimation, spec, remove_identity, IDENTITY,
                                                DRAWS, seed, BATCH, AT_TARGET, **options)
    return clean, counter, info


def effect_rows(named_runs, full_pair, floor_pair, matches, label_prefix=""):
    rows = []
    for behaviour, pick in [("identity effect", lambda c, k: c - k), ("arbitration", lambda c, k: c)]:
        full_value, floor_value = pick(*full_pair), pick(*floor_pair)
        for name, (clean, counter) in named_runs.items():
            value = pick(clean, counter)
            share = scrub.recovered_share(value, full_value, floor_value, matches, N_BOOT)
            rows.append({"behaviour": behaviour, "run": label_prefix + name, "recovered share": share["recovery"],
                         "CI low": share["ci_low"], "CI high": share["ci_high"], "mean value": float(value.mean()),
                         "mean full": float(full_value.mean()), "mean floor": float(floor_value.mean()), "n": len(value)})
    return rows


runs_corrected, scrub_corrected = {}, pd.DataFrame()
if RUN:
    start = time.time()
    matches = estimation["match_id"].to_numpy()
    hypothesis_spec = {**{c: CLASS_KEYS.get(c) for c in CIRCUIT}, **{c: None for c in OUTSIDE}}
    full_pair = (clean_intact, corrupt_intact)
    floor_clean, floor_counter, _ = run({c: None for c in ALL}, 30)
    floor2_clean, floor2_counter, _ = run({c: None for c in ALL}, 31)
    hyp_clean, hyp_counter, hyp_info = run(hypothesis_spec, 10)
    same_clean, same_counter, _ = run(hypothesis_spec, 10, branch_rule="all_same")
    removed_clean, removed_counter, _ = run({c: None for c in CIRCUIT}, 20)
    runs_corrected = {"hypothesis (paired donors, corrected)": (hyp_clean, hyp_counter),
                      "hypothesis, original same-donor rule, identical donors": (same_clean, same_counter),
                      "circuit scrubbed": (removed_clean, removed_counter),
                      "floor, second independent draw (noise check)": (floor2_clean, floor2_counter)}
    scrub_corrected = pd.DataFrame(effect_rows(runs_corrected, full_pair, (floor_clean, floor_counter), matches))
    noise = scrub.recovery_noise_check(floor_clean - floor_counter, floor2_clean - floor2_counter, clean_intact - corrupt_intact, matches, N_BOOT)
    print(f"corrected runs done in {time.time() - start:.0f} s | branch modes: {hyp_info['modes']}")
    print(scrub_corrected.round(4).to_string(index=False))
    print("\nnoise check (identity effect):", {k: round(v, 4) for k, v in noise.items()})
    scrub_corrected.to_csv(OUT / "results" / "MB11b_scrub_corrected.csv", index=False)

corrected runs done in 3 s | branch modes: {'L0H0': 'paired', 'L0H1': 'paired', 'L0MLP': 'paired', 'L1H0': 'same', 'L1H1': 'same', 'L1MLP': 'paired', 'L2H0': 'paired', 'L2H1': 'paired', 'L2MLP': 'paired', 'L3H1': 'same', 'L3MLP': 'paired', 'L3H0': 'same'}
      behaviour                                                    run  recovered share  CI low  CI high  mean value  mean full  mean floor   n
identity effect                  hypothesis (paired donors, corrected)           0.7536  0.6647   0.8206     -0.1629    -0.1672     -0.0097 529
identity effect hypothesis, original same-donor rule, identical donors          -0.0000 -0.0103   0.0108     -0.0100    -0.1672     -0.0097 529
identity effect                                       circuit scrubbed           0.0039  0.0009   0.0075     -0.0083    -0.1672     -0.0097 529
identity effect           floor, second independent draw (noise check)           0.0014 -0.0019   0.0056     -0.0099    -0.1672     -0.0097 529
    arbitration         

### R3 — Wrong-class and unrelated-class controls, corrected (F04)

The scoring table (`a_ctx`, `a_id`, windows, positions) is never modified; this is asserted by fingerprint. Each draw permutes the rows once and
gives every keyed component the **complete** label tuple of the permuted row. The number of classes, every class size and
every joint tuple frequency are therefore preserved, and only the pairing between events and labels is broken. Diagnostics
report how often a wrong-class donor still shares the recipient's true class. The unrelated-class control scrubs every
circuit component within a class built from a field the abstraction does not name (`team`, as in Section 5).

In [13]:
control_rows_corrected, wrong_diagnostics = [], []
if RUN:
    full_effect = clean_intact - corrupt_intact
    floor_effect = floor_clean - floor_counter
    hypothesis_effect = hyp_clean - hyp_counter
    wrong_clean, wrong_counter, wrong_info = run(hypothesis_spec, 40, wrong_class=True)
    wrong_diagnostics = wrong_info["wrong_class_diagnostics"]
    alternatives = {"wrong class (complete labels permuted)": (wrong_clean, wrong_counter)}
    spare = [k for k in CLASS_KEYS_AVAILABLE if k not in {k for keys in CLASS_KEYS.values() for k in keys}]
    if spare:
        unrelated_clean, unrelated_counter, _ = run({**{c: [spare[0]] for c in CIRCUIT}, **{c: None for c in OUTSIDE}}, 50)
        alternatives[f"team-keyed invariant-branch sensitivity ({spare[0]}; classes and branch rule both changed)"] = (unrelated_clean, unrelated_counter)
    for name, (clean, counter) in alternatives.items():
        value = clean - counter
        share = scrub.recovered_share(value, full_effect, floor_effect, matches, N_BOOT)
        difference = mi_metrics.aligned(hypothesis_effect - value, full_effect - floor_effect)
        summary = mi_metrics.summarise(difference, matches, N_BOOT)
        control_rows_corrected.append({"control": name, "recovered share": share["recovery"], "CI low": share["ci_low"], "CI high": share["ci_high"],
                                       "hypothesis minus this (aligned mean)": summary["mean"], "difference CI low": summary["ci_low"],
                                       "difference CI high": summary["ci_high"],
                                       "one-sided p (hypothesis greater)": mi_metrics.bootstrap_pvalue(difference, matches, N_BOOT, alternative="greater")})
scrub_controls_corrected = pd.DataFrame(control_rows_corrected)
scrub_controls_corrected.to_csv(OUT / "results" / "MB11b_controls_corrected.csv", index=False)
print(scrub_controls_corrected.round(4).to_string(index=False))
if wrong_diagnostics:
    print("\nwrong-class diagnostics (first draw):")
    print(pd.DataFrame(wrong_diagnostics[0]).T.round(3).to_string())

                                                                             control  recovered share  CI low  CI high  hypothesis minus this (aligned mean)  difference CI low  difference CI high  one-sided p (hypothesis greater)
                                              wrong class (complete labels permuted)           0.1978  0.1623   0.2409                                0.2544             0.1837              0.3170                            0.0005
team-keyed invariant-branch sensitivity (team; classes and branch rule both changed)          -0.0003 -0.0037   0.0026                                0.3451             0.2554              0.4216                            0.0005

wrong-class diagnostics (first draw):
      classes class sizes preserved share of donors that share the recipient's true class share of rows that are their own donor (singleton)
L0H0      243                  True                                              0.234405                                         

### R4 — Which node's abstraction is load-bearing, corrected

In [14]:
node_rows_corrected = []
if RUN and len(CIRCUIT) > 1:
    denominator = full_effect - floor_effect
    for index, component in enumerate(CIRCUIT):
        spec = dict(hypothesis_spec); spec[component] = None
        plan = [{c: values.copy() for c, values in assignment.items()} for assignment in hyp_info["assignments"]]
        for draw, assignment in enumerate(plan):
            assignment[component] = circuits.class_donors([0] * len(estimation), np.random.default_rng(6000 + 100 * index + draw))
            for unaffected in set(ALL) - {component}:
                assert np.array_equal(assignment[unaffected], hyp_info["assignments"][draw][unaffected])
        clean, counter, info = run(spec, 60, assignment_plan=plan, modes_override=hyp_info["modes"])
        share = scrub.recovered_share(clean - counter, full_effect, floor_effect, matches, N_BOOT)
        drop = mi_metrics.pooled_recovery(hypothesis_effect - (clean - counter), denominator, matches, N_BOOT)
        draw_drops = [mi_metrics.pooled_ratio((hyp_info["draw_clean"][d] - hyp_info["draw_counter"][d]) - (info["draw_clean"][d] - info["draw_counter"][d]), denominator) for d in range(DRAWS)]
        node_rows_corrected.append({"node scrubbed freely": component, "class keys it loses": ", ".join(CLASS_KEYS.get(component, [])),
            "recovered share without its class": share["recovery"], "recovery CI low": share["ci_low"], "recovery CI high": share["ci_high"],
            "drop from the full hypothesis": drop["recovery"], "paired drop CI low": drop["ci_low"], "paired drop CI high": drop["ci_high"],
            "draw-level drop SD (Monte Carlo diagnostic)": float(np.std(draw_drops, ddof=1)) if DRAWS > 1 else np.nan,
            "unaffected donors held fixed": True, "branch rule preserved": True,
            "share of events in a singleton class": singleton.get(component, np.nan)})
per_node_corrected = pd.DataFrame(node_rows_corrected)
per_node_corrected.to_csv(OUT / "results" / "MB11b_per_node_corrected.csv", index=False)
print(per_node_corrected.round(4).to_string(index=False))


node scrubbed freely        class keys it loses  recovered share without its class  recovery CI low  recovery CI high  drop from the full hypothesis  paired drop CI low  paired drop CI high  draw-level drop SD (Monte Carlo diagnostic)  unaffected donors held fixed  branch rule preserved  share of events in a singleton class
                L0H0 a_ctx, team_change, player                             0.7546           0.6649            0.8209                        -0.0010             -0.0035               0.0014                                       0.0007                          True                   True                                0.2287
                L0H1                     player                             0.7529           0.6632            0.8191                         0.0007             -0.0004               0.0025                                       0.0011                          True                   True                                0.0000
               L0MLP a

### R5 — Post-hoc competing hypotheses (F07)

`config/mechanism_hypothesis.json` stays as written, with empty sentences. The hypotheses below are **post-hoc**. They
paraphrase the competing hypotheses MB16 wrote in `config/MB16_plan.json` after MB12 had opened the test split, and they are
stored in a separate file whose `status` says so. Each states what each component computes and which prediction separates
it from the others. They are tested here with the corrected scrub, on the validation estimation half only.

| Hypothesis | Claim | Distinguishing prediction |
|---|---|---|
| H0 default (historical) | the class keys in `mechanism_hypothesis.json`; **no computation stated** | none can be stated: the keys were derived from which MB06 contrast selected each node |
| H1 shared output-key hypothesis | each MLP's output at the scored event depends only on (player, role, previous action, possession change, coupled flag); heads carry nothing about the behaviour | keeps the identity effect **and** Δ, and beats its wrong-class control on the identity effect |
| H2 player-blind output-key hypothesis | as H1 without the player: the MLPs do not read the player | keeps Δ but **loses** the identity effect, falling to its wrong-class control |
| H3 heads matter | each head's output depends only on H1's keys; MLPs carry nothing | loses Δ (MLPs free) |

H1 and H2 differ only in whether the player is in the MLPs' keys. With paired donors that difference changes the branch rule
of every MLP, so the two hypotheses make different predictions about the identity effect.

In [15]:
POSTHOC_KEYS_FULL = ["player", "role", "prev_action_id", "team_change", "coupled_with_prev"]
POSTHOC_KEYS_BLIND = ["role", "prev_action_id", "team_change", "coupled_with_prev"]
posthoc_rows = []
posthoc_path = OUT / "config" / "mechanism_hypotheses_posthoc.json"
if RUN and all(key in estimation.columns for key in POSTHOC_KEYS_FULL):
    HEADS_ALL = [c for c in ALL if not c.endswith("MLP")]
    MLPS_ALL = [c for c in ALL if c.endswith("MLP")]
    posthoc = {
        "status": "POST-HOC. Written 2026-10-04 by the repair, after MB12's test access and after MB16. Paraphrases MB16_plan.json. "
                  "NOT pre-registered; validation estimation half only; must not be read against MB12's test results.",
        "hypotheses": {
            "H0 default (historical)": {"computes": "NOT STATED: historical default class keys of config/mechanism_hypothesis.json",
                                        "spec": {c: CLASS_KEYS.get(c) for c in CIRCUIT}, "free": OUTSIDE},
            "H1 shared output-key hypothesis": {"computes": "each MLP output at the scored event is a function of (player, role, previous action, "
                                            "possession change, coupled flag) only; head outputs carry nothing about the behaviour",
                                "spec": {**{c: POSTHOC_KEYS_FULL for c in MLPS_ALL}, **{c: None for c in HEADS_ALL}}},
            "H2 player-blind output-key hypothesis": {"computes": "each MLP output is a function of (role, previous action, possession change, coupled flag) "
                                                  "only and does not read the player; heads carry nothing",
                                      "spec": {**{c: POSTHOC_KEYS_BLIND for c in MLPS_ALL}, **{c: None for c in HEADS_ALL}}},
            "H3 heads matter": {"computes": "each head output is a function of H1's keys only; MLP outputs carry nothing about the behaviour",
                                "spec": {**{c: POSTHOC_KEYS_FULL for c in HEADS_ALL}, **{c: None for c in MLPS_ALL}}}},
        "discriminating_predictions": {"H1 vs H2": "identity effect: H1 above its wrong-class control, H2 not; both keep Delta",
                                       "H1 vs H3": "Delta: H1 keeps it, H3 does not"},
        "scrub": "src/scrub.py, branch_rule 'hypothesis' (paired counterfactual donors), at the scored position",
        "written_at": fingerprint.now()}
    if not posthoc_path.exists():
        mi_setup.save_json(posthoc_path, posthoc)
    posthoc = mi_setup.load_json(posthoc_path)
    fingerprint.record_files(MANIFEST, {"config/mechanism_hypotheses_posthoc.json": posthoc_path})
    for index, (name, entry) in enumerate(posthoc["hypotheses"].items()):
        spec = {**entry["spec"], **{c: None for c in entry.get("free", [])}}
        variant_runs = {}
        for variant, options in [("right class", {}), ("wrong class", {"wrong_class": True})]:
            variant_runs[variant] = run(spec, 100 + 2 * index + (variant == "wrong class"), **options)
        common_nonself = np.logical_and(variant_runs["right class"][2]["nonself_every_keyed_node_all_draws"],
                                       variant_runs["wrong class"][2]["nonself_every_keyed_node_all_draws"])
        for variant, (clean, counter, info) in variant_runs.items():
            fully = common_nonself
            for subset, keep in [("all estimation events", np.ones(len(estimation), dtype=bool)), ("common events: actual non-self donor at every keyed node in all draws of both variants", fully)]:
                for behaviour, pick in [("identity effect", lambda c, k: c - k), ("arbitration", lambda c, k: c)]:
                    share = scrub.recovered_share(pick(clean, counter)[keep], pick(clean_intact, corrupt_intact)[keep],
                                                  pick(floor_clean, floor_counter)[keep], matches[keep], N_BOOT)
                    posthoc_rows.append({"hypothesis": name, "donors": variant, "events": subset, "n": int(keep.sum()), "behaviour": behaviour,
                                         "recovered share": share["recovery"], "CI low": share["ci_low"], "CI high": share["ci_high"]})
posthoc_results = pd.DataFrame(posthoc_rows)
posthoc_results.to_csv(OUT / "results" / "MB11b_posthoc_hypotheses_corrected.csv", index=False)
if len(posthoc_results):
    print(posthoc_results.pivot_table(index=["events", "hypothesis", "donors"], columns="behaviour", values="recovered share").round(3).to_string())

behaviour                                                                                                                                 arbitration  identity effect
events                                                                                 hypothesis                            donors                                   
all estimation events                                                                  H0 default (historical)               right class        0.978            0.763
                                                                                                                             wrong class        0.230            0.242
                                                                                       H1 shared output-key hypothesis       right class        0.935            0.870
                                                                                                                             wrong class        0.417            0.48

In [16]:
summary_corrected = {"status": "REPAIRED 2026-10-04 (F02, F03, F04, F07); validation estimation half; not part of the MB12 lock",
                     "self_scrub": check if RUN else None, "scrub": scrub_corrected.to_dict(orient="records"),
                     "noise_check": noise if RUN else None, "controls": scrub_controls_corrected.to_dict(orient="records"),
                     "wrong_class_diagnostics_first_draw": wrong_diagnostics[0] if wrong_diagnostics else None,
                     "per_node": per_node_corrected.to_dict(orient="records"),
                     "posthoc": posthoc_results.to_dict(orient="records"), "class_keys": CLASS_KEYS, "draws": DRAWS, "run_mode": RUN_MODE}
mi_setup.save_json(OUT / "results" / "MB11b_summary_corrected.json", summary_corrected)
fingerprint.record_files(MANIFEST, {"results/MB11b_summary_corrected.json": OUT / "results" / "MB11b_summary_corrected.json"})
fingerprint.register(REGISTRY, {"notebook": "MB11b", "experiment": "R: corrected paired causal scrub (F02-F04) + post-hoc hypotheses (F07)",
                                "seed": PRIMARY_SEED, "run_mode": RUN_MODE})
print("saved")

saved


**Reading Section R (fixed before the corrected numbers exist).**

* R1 must pass. The original same-donor design must fail invariant 2 whenever components respond to the player.
* The noise check must give a recovered share near 0. If it does not, the identity-effect shares in R2 to R5 cannot be read.
* An abstraction is supported for a behaviour only if its recovered-share CI lies **above** its wrong-class control's CI
  (R3 for H0; R5 for H1 to H3). A high share that its wrong-class control matches says only how much resampling at the
  scored position leaves Δ unchanged.
* H0 has no stated computation. Whatever it scores, it tests class keys, not a mechanism.
* Nothing here was pre-registered, and MB12's test results (including M4 with the defective design) were seen first.

**Researcher interpretation of Section R** *(to be written after reading the corrected numbers)*

- What was tested:
- What was observed:
- What it supports / contradicts:
- What it does not establish:

R10–R13: filtered comparisons use the intersection of actual donor-feasible rows across right/wrong variants and all draws. Per-node comparisons freeze all other donor assignments and branch modes, with a paired drop interval and a separate donor-draw variability diagnostic. Intervals condition on the sampled donor plans. This experiment resamples target outputs from intact donor forwards: it tests output-equivalence classes, not a recursively implemented serial algorithm. H2 enforces player invariance through its branch rule; near-zero identity recovery alone does not discover player blindness. Hypotheses remain exploratory.